# Circuit Info Table

Gathers the compiled **ansatz-only** costs (bare state-preparation circuit, no Hamiltonian-measurement rotations) computed by `../ignore/save_ansatz_only.py` and saved as `ansatz_only_result.json` in every H1-1E / H1-Emulator circuit directory, into a single table -- one row per circuit (not per sample). The circuits are compiled with the H1 compiler (`QuantinuumBackend.get_compiled_circuit`, optimisation level 2, or 0 for `0_givens`), so gates are in the native PhasedX / ZZPhase set. Total gates = 1-qubit + 2-qubit gates.

In [5]:
import json
import os
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent
GIVENS_DIR = BASE_DIR / "Givens_results"
UCCSD_DIR = BASE_DIR / "UCCSD_results"
ADAPT_DIR = BASE_DIR / "ADAPT-VQE_results"
OUTPUT_CSV = BASE_DIR / "figures" / "circuit_info_table.csv"

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

In [ ]:
def nb_params_from(data):
    """Parameter count from the VQE result (ansatz_only_result.json doesn't always store it)."""
    nb_params = data.get("nb_params")
    if nb_params is not None:
        return nb_params
    final_params = data.get("final_params", [])
    if final_params and isinstance(final_params[0], list):
        return len(final_params[0])
    return len(final_params)


def row_from_circuit_dir(circuit_dir: Path, ansatz: str, backend: str, molecule: str):
    ansatz_path = circuit_dir / "ansatz_only_result.json"
    if not ansatz_path.exists():
        return None
    with open(ansatz_path) as f:
        a = json.load(f)

    vqe_path = circuit_dir / "vqe_result.json"
    if not vqe_path.exists():
        vqe_path = circuit_dir / "temp_vqe_result.json"
    with open(vqe_path) as f:
        vqe = json.load(f)

    return {
        "ansatz": ansatz,
        "backend": backend,
        "molecule": molecule,
        "circuit": a.get("circuit", circuit_dir.name),
        "nb_params": nb_params_from(vqe),
        "nb_qubits": a.get("n_qubits"),
        "n_1q_gates": a["n_1q_gates"],
        "n_2q_gates": a["n_2q_gates"],
        "total_gates": a["n_1q_gates"] + a["n_2q_gates"],
        "depth": a["depth"],
        "depth_2q": a["depth_2q"],
        "n_rz_gates": a["n_rz_gates"],
        "optimisation_level": a.get("optimisation_level"),
    }


def collect_leaf_results(backend_dir: Path, ansatz: str, backend_label: str):
    """One row per circuit directory directly under backend_dir/<molecule>/<circuit>.

    Does not descend into sample_* subdirectories: samples of the same circuit
    share the same ansatz, so only the circuit-level ansatz_only_result.json is read.
    """
    rows = []
    if not backend_dir.exists():
        return rows
    for molecule_dir in sorted(p for p in backend_dir.iterdir() if p.is_dir()):
        for circuit_dir in sorted(p for p in molecule_dir.iterdir() if p.is_dir()):
            row = row_from_circuit_dir(circuit_dir, ansatz, backend_label, molecule_dir.name)
            if row is not None:
                rows.append(row)
    return rows

In [ ]:
sources = [
    (GIVENS_DIR / "H1-1E", "Givens", "H1-1E"),
    (UCCSD_DIR / "H1-1E", "UCCSD", "H1-1E"),
    (ADAPT_DIR / "uccsd" / "H1-Emulator_results", "ADAPT-VQE (UCC pool)", "H1-Emulator"),
    (ADAPT_DIR / "generalized" / "H1-Emulator_results", "ADAPT-VQE (generalised pool)", "H1-Emulator"),
]

rows = []
for backend_dir, ansatz, backend_label in sources:
    rows.extend(collect_leaf_results(backend_dir, ansatz, backend_label))

df = pd.DataFrame(rows).sort_values(["ansatz", "molecule", "nb_params"]).reset_index(drop=True)
print(f"{len(df)} circuits found")
df

98 circuits found


,ansatz,backend,molecule,circuit,nb_params,nb_qubits,n_1q_gates,n_2q_gates,total_gates,depth,depth_2q,n_rz_gates,optimisation_level
0,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_01_params,1,8,63,33,96,66,33,1,2
1,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_02_params,2,8,140,93,233,186,93,5,2
2,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_03_params,3,8,143,95,238,190,95,4,2
3,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_04_params,4,8,154,100,254,192,96,5,2
4,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_05_params,5,8,257,186,443,364,182,7,2
5,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_06_params,6,8,339,258,597,508,254,7,2
6,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_07_params,7,8,427,334,761,660,330,7,2
7,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_08_params,8,8,442,340,782,672,336,7,2
8,ADAPT-VQE (generalised pool),H1-Emulator,BeH2,adapt_09_params,9,8,455,349,804,674,337,6,2
9,ADAPT-VQE (generalised pool),H1-Emulator,LiH_6_spinorbs,adapt_01_params,1,6,60,33,93,66,33,1,2


In [8]:
OUTPUT_CSV.parent.mkdir(exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_CSV}")

Saved 98 rows to /home/qlila/ResearchCode/QMC_Givens_ansatz_results/figures/circuit_info_table.csv
